**Employee_Attendance_and_Productivity_Tracker**

In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Employee_Attendance_and_Productivity_Tracker") \
    .getOrCreate()
spark

In [23]:
from google.colab import files

uploaded = files.upload()

Saving employee_attendance.csv to employee_attendance.csv


In [25]:
# Loading Dataset
df = spark.read.csv("employee_attendance.csv",header=True,inferSchema=True)
print("Total Records:", df.count())
df.printSchema()
df.show(5)

Total Records: 10000
root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- attendance_date: date (nullable = true)
 |-- clock_in: timestamp (nullable = true)
 |-- clock_out: timestamp (nullable = true)
 |-- attendance_status: string (nullable = true)
 |-- work_hours: double (nullable = true)
 |-- productivity_score: double (nullable = true)

+-----------+-------------+----------+---------------+-------------------+-------------------+-----------------+----------+------------------+
|employee_id|employee_name|department|attendance_date|           clock_in|          clock_out|attendance_status|work_hours|productivity_score|
+-----------+-------------+----------+---------------+-------------------+-------------------+-----------------+----------+------------------+
|       1103|Employee_1103|Operations|     2026-05-20|2026-06-17 08:05:00|2026-06-17 16:26:00|          Present|      8.35|             78.9

In [26]:
# Late Logins
late_logins = df.filter(col("clock_in") > "09:30:00")
print("LATE LOGIN EMPLOYEES")
late_logins.show(truncate=False)

LATE LOGIN EMPLOYEES
+-----------+-------------+----------+---------------+-------------------+-------------------+-----------------+----------+------------------+
|employee_id|employee_name|department|attendance_date|clock_in           |clock_out          |attendance_status|work_hours|productivity_score|
+-----------+-------------+----------+---------------+-------------------+-------------------+-----------------+----------+------------------+
|1180       |Employee_1180|Finance   |2026-02-18     |2026-06-17 10:11:00|2026-06-17 17:55:24|Present          |7.74      |79.67             |
|1122       |Employee_1122|Admin     |2026-02-13     |2026-06-17 10:09:00|2026-06-17 16:40:12|Present          |6.52      |51.59             |
|1088       |Employee_1088|Finance   |2026-01-17     |2026-06-17 10:35:00|2026-06-17 18:27:12|Present          |7.87      |78.45             |
|1053       |Employee_1053|Finance   |2026-01-01     |2026-06-17 09:37:00|2026-06-17 17:59:12|Present          |8.37     

In [27]:
# Absent Employees
absent_employees = df.filter(col("attendance_status") == "Absent")
print("ABSENT EMPLOYEES")
absent_employees.show(truncate=False)

ABSENT EMPLOYEES
+-----------+-------------+----------+---------------+--------+---------+-----------------+----------+------------------+
|employee_id|employee_name|department|attendance_date|clock_in|clock_out|attendance_status|work_hours|productivity_score|
+-----------+-------------+----------+---------------+--------+---------+-----------------+----------+------------------+
|1022       |Employee_1022|Marketing |2026-06-03     |NULL    |NULL     |Absent           |0.0       |0.0               |
|1190       |Employee_1190|Sales     |2026-05-01     |NULL    |NULL     |Absent           |0.0       |0.0               |
|1132       |Employee_1132|HR        |2026-01-24     |NULL    |NULL     |Absent           |0.0       |0.0               |
|1035       |Employee_1035|Operations|2026-03-26     |NULL    |NULL     |Absent           |0.0       |0.0               |
|1081       |Employee_1081|Operations|2026-01-10     |NULL    |NULL     |Absent           |0.0       |0.0               |
|1054  

In [28]:
# Department Summary
department_summary = df.groupBy("department").agg(round(avg("work_hours"), 2).alias("avg_work_hours"),round(avg("productivity_score"), 2).alias("avg_productivity_score"))
print("DEPARTMENT SUMMARY")
department_summary.show(truncate=False)

DEPARTMENT SUMMARY
+----------+--------------+----------------------+
|department|avg_work_hours|avg_productivity_score|
+----------+--------------+----------------------+
|Sales     |7.24          |67.31                 |
|HR        |7.21          |66.75                 |
|Finance   |7.25          |68.34                 |
|Admin     |7.17          |67.44                 |
|Marketing |7.17          |66.95                 |
|IT        |7.13          |67.85                 |
|Support   |7.09          |67.21                 |
|Operations|7.12          |66.63                 |
+----------+--------------+----------------------+



In [29]:
# Attendance Issues by Department
attendance_issues = df.groupBy("department").agg(sum(when(col("attendance_status") == "Absent", 1) .otherwise(0)).alias("absent_count"),
    sum(when(col("clock_in") > "09:30:00", 1).otherwise(0)).alias("late_login_count"))
print("ATTENDANCE ISSUES BY DEPARTMENT")
attendance_issues.show(truncate=False)

ATTENDANCE ISSUES BY DEPARTMENT
+----------+------------+----------------+
|department|absent_count|late_login_count|
+----------+------------+----------------+
|Sales     |123         |578             |
|HR        |130         |585             |
|Finance   |116         |549             |
|Admin     |124         |591             |
|Marketing |136         |568             |
|IT        |129         |577             |
|Support   |138         |560             |
|Operations|133         |517             |
+----------+------------+----------------+



In [30]:
# Save Outputs
attendance_issues.coalesce(1).write.mode("overwrite").option("header", True).csv("attendance_issues_by_department")